In [1]:
import json
import torch
from tqdm import tqdm
from torch import nn
from torch.optim import AdamW
from typing import List, Dict
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import DataLoader
from data_process import KanbunDataset, kanbun_collate_fn, get_label_map, decode_prediction, character_mark
from model import BaseKanbunModel, OneAuxiliaryTaskModel, TwoAuxiliaryTaskModel, ThreeAuxiliaryTaskModel, FourAuxiliaryTaskModel
from translation import valid_marks, translate_corpus
from evaluation import compute_bleu_score, compute_chrf_score, compute_bert_score, compute_rouge_score, compute_ribes_score, compute_ter_score, compute_kendalltau_score, compute_pmr_score

/root/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
# This code is used for internet connection. If you don't use proxy, please ignore it.
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [3]:
l2i, i2l = get_label_map(f"./labels.json")

In [4]:
device = torch.device("cuda")
epoches = 15
batch_size = 64
learning_rate = 5e-5
weight_decay = 0.01

In [5]:
def evaluate(model, data):
    model.eval()
    with torch.no_grad():
        n = 0
        l = 0
        for _, (sentences, labels) in tqdm(enumerate(data), total = len(data)):
            loss, logits_okurigana, logits_particle, logits_position = model(sentences, labels)
            l += loss.item()
            n += logits_okurigana.size(0)
        print("Evaluate loss {}".format(l / n))
    return l / n

def predict(model, data):
    model.eval()
    with torch.no_grad():
        okurigana = []
        particle = []
        position = []
        for _, (sentences, _) in tqdm(enumerate(data), total = len(data)):
            _, logits_okurigana, logits_particle, logits_position = model(sentences, None)
            probabilities_okurigana = nn.functional.softmax(logits_okurigana, dim = 1)
            probabilities_particle = nn.functional.softmax(logits_particle, dim = 1)
            probabilities_position = nn.functional.softmax(logits_position, dim = 1)
            okurigana.extend(torch.argmax(probabilities_okurigana, dim = 1).tolist())
            particle.extend(torch.argmax(probabilities_particle, dim = 1).tolist())
            position.extend(torch.argmax(probabilities_position, dim = 1).tolist())
    return [okurigana, particle, position]

def train(model, train_data, dev_data, optimizer, epoches):
    model.train(True)
    dev_loss = 10
    for e in range(epoches):
        n = 0
        l = 0
        for _, (sentences, labels) in tqdm(enumerate(train_data), total = len(train_data)):
            optimizer.zero_grad()
            loss, logits_okurigana, logits_particle, logits_position = model(sentences, labels)
            loss.backward()
            optimizer.step()
            l += loss.item()
            n += logits_okurigana.size(0)
        print("=" * 20)
        print("Epoch {}".format(e + 1))
        print("Traning loss {}".format(l / n))
    
        current_dev_loss = evaluate(model, dev_data)
        if current_dev_loss <= dev_loss:
            dev_loss = current_dev_loss
        else:
            break

In [6]:
tokenizer = AutoTokenizer.from_pretrained("KoichiYasuoka/roberta-classical-chinese-base-char")
bert = AutoModel.from_pretrained("KoichiYasuoka/roberta-classical-chinese-base-char")
loss_function = nn.CrossEntropyLoss()
model = TwoAuxiliaryTaskModel(bert, loss_function, 768, [218, 184, 11, 2, 38], ["segmentation", "dependencytype"], (0.7, 0.3), 0.3).to(device)
optimizer = AdamW(model.parameters(), lr = learning_rate, weight_decay = weight_decay)

In [7]:
train_data = KanbunDataset(f"./Kanbun Data/train.json", "japanese", ["okurigana", "particle", "position", "segmentation", "partofspeech", "dependencyarc", "dependencytype"], tokenizer, l2i, device = device)
dev_data = KanbunDataset(f"./Kanbun Data/val.json", "japanese", ["okurigana", "particle", "position", "segmentation", "partofspeech", "dependencyarc", "dependencytype"], tokenizer, l2i, device = device)
test_data = KanbunDataset(f"./Kanbun Data/test.json", "japanese", ["okurigana", "particle", "position", "segmentation", "partofspeech", "dependencyarc", "dependencytype"], tokenizer, l2i, device = device)
train_loader = DataLoader(dataset = train_data, batch_size = batch_size, shuffle = True, collate_fn = lambda batch:kanbun_collate_fn(batch, tokenizer.pad_token_id))
dev_loader = DataLoader(dataset = dev_data, batch_size = batch_size, shuffle = True, collate_fn = lambda batch:kanbun_collate_fn(batch, tokenizer.pad_token_id))
test_loader = DataLoader(dataset = test_data, batch_size = batch_size, shuffle = False, collate_fn = lambda batch:kanbun_collate_fn(batch, tokenizer.pad_token_id))

In [8]:
train(model, train_loader, dev_loader, optimizer, epoches)

100%|██████████| 43/43 [00:04<00:00,  9.36it/s]


Epoch 1
Traning loss 0.08811725775400797


100%|██████████| 5/5 [00:00<00:00, 18.92it/s]


Evaluate loss 0.061620354652404785


100%|██████████| 43/43 [00:04<00:00, 10.30it/s]


...

Evaluate loss 0.024583344161510468


100%|██████████| 43/43 [00:04<00:00, 10.51it/s]


Epoch 14
Traning loss 0.006165183125398098


100%|██████████| 5/5 [00:00<00:00, 18.20it/s]

Evaluate loss 0.024797795712947844


In [9]:
prediction = predict(model, test_loader)
result = character_mark(f"./Kanbun Data/test.json", decode_prediction(f"./Kanbun Data/test.json", prediction, i2l))

100%|██████████| 6/6 [00:00<00:00, 20.90it/s]


In [10]:
original_marks, valid_marks, _ = valid_marks(f"./Kanbun Data/test.json", result)
original_sentences, translated_sentences, valid_count = translate_corpus(original_marks, valid_marks)

In [11]:
compute_bleu_score(original_sentences, translated_sentences)

BLEU = 56.09 79.4/62.0/49.9/40.3 (BP = 1.000 ratio = 1.089 hyp_len = 3720 ref_len = 3417)

In [12]:
compute_chrf_score(original_sentences, translated_sentences)

chrF2 = 52.54

In [13]:
compute_bert_score(original_sentences, translated_sentences)

(0.9234816058207367, 0.9328659884000229, 0.9280617344850874)

In [14]:
compute_rouge_score(original_sentences, translated_sentences)

{'rouge1': {'precision': 0.8717827605539463,
  'recall': 0.8113348737712486,
  'fmeasure': 0.8356393040485233},
 'rouge2': {'precision': 0.6945079541041652,
  'recall': 0.6484655692512384,
  'fmeasure': 0.6669744361338176},
 'rougeL': {'precision': 0.8344784766394937,
  'recall': 0.7779289345515009,
  'fmeasure': 0.8008848367777631}}

In [15]:
compute_ribes_score(original_sentences, translated_sentences)

0.6294555190079184

In [16]:
compute_ter_score(original_sentences, translated_sentences)

Using the latest cached version of the module from /root/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--ter/9c9af3214842a93c26c9ac10ddc5d07559df8e38c0ab3b599e8121b9aae196bd (last modified on Wed Dec 31 19:07:38 2025) since it couldn't be found locally at evaluate-metric--ter, or remotely on the Hugging Face Hub.


{'score': 26.989247311827956, 'num_edits': 1004, 'ref_length': 3720.0}

In [17]:
compute_kendalltau_score(original_marks, valid_marks)

np.float64(0.928410008071025)

In [18]:
compute_pmr_score(original_marks, valid_marks)

0.900322841000807

In [19]:
len(original_marks) / len(result)

0.9567567567567568

In [20]:
print(original_sentences[10])

古木寒鳥鳴き


In [21]:
print(translated_sentences[10])

古木寒鳥を鳴く


In [22]:
print(original_sentences[15])

季布二諾無く


In [23]:
print(translated_sentences[15])

季布二諾無く
